# RAW-RAG: tiny raw-history retrieval demo

This notebook accompanies [Bao et al., arXiv:2610.30755](https://arxiv.org/abs/2610.30755). It preserves raw timestamped turns, performs dense retrieval, reranks with a cross-encoder, and expands the selected windows back to complete turns. The models and retrieval depths are deliberately smaller than the paper's so the demo runs on a Colab CPU.

In [ ]:
!pip -q install sentence-transformers transformers

import json
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer

## 1. Load one source-preserving conversation

When uploaded beside this notebook, `examples.json` is used. The embedded fallback makes the notebook runnable immediately in Colab. The record is synthetic; this paper introduces a method rather than a new dataset.

In [ ]:
fallback = [{
    'conversation_id': 'synthetic_alex_001',
    'synthetic': True,
    'messages': [
        {'timestamp': '2026-01-04T09:00:00Z', 'speaker': 'user', 'content': 'I live in Basel and commute to Zurich twice a week.'},
        {'timestamp': '2026-01-04T09:01:00Z', 'speaker': 'assistant', 'content': 'That is useful context for future travel suggestions.'},
        {'timestamp': '2026-02-12T18:30:00Z', 'speaker': 'user', 'content': 'My favorite way to travel is by train, and I prefer quiet carriages.'},
        {'timestamp': '2026-03-20T12:00:00Z', 'speaker': 'user', 'content': 'I accepted a new job in Lausanne. I will start in May.'},
        {'timestamp': '2026-04-28T16:45:00Z', 'speaker': 'user', 'content': 'Update: I found an apartment in Lausanne near Ouchy and will move there this weekend.'},
        {'timestamp': '2026-05-03T10:15:00Z', 'speaker': 'assistant', 'content': 'Congratulations on the move to Lausanne.'},
        {'timestamp': '2026-06-08T07:40:00Z', 'speaker': 'user', 'content': 'The new commute is short, so I usually cycle to work now. I still take trains for longer trips.'}
    ],
    'question': 'Where does the user currently live, and how do they usually commute to work?',
    'expected_answer': 'The user currently lives in Lausanne near Ouchy and usually cycles to work.'
}]

path = Path('examples.json')
records = json.loads(path.read_text()) if path.exists() else fallback
example = records[0]
print(json.dumps(example, indent=2))

## 2. Serialize and chunk raw history

The paper uses 256-token windows with 32-token overlap. We use 64 and 16 here so the tiny conversation yields several windows. Each window keeps the indexes of overlapping source turns, enabling complete-turn expansion after retrieval.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def serialize_turn(message):
    return f"[{message.get('timestamp', 'unknown')}] {message['speaker']}: {message['content']}"

turns = [serialize_turn(m) for m in example['messages']]
turn_token_ids = [tokenizer.encode(t, add_special_tokens=False) for t in turns]
flat_ids, owners = [], []
for turn_index, ids in enumerate(turn_token_ids):
    flat_ids.extend(ids)
    owners.extend([turn_index] * len(ids))

def make_windows(token_ids, token_owners, size=64, overlap=16):
    windows = []
    for start in range(0, len(token_ids), size - overlap):
        end = min(start + size, len(token_ids))
        owner_ids = sorted(set(token_owners[start:end]))
        windows.append({
            'window_id': len(windows),
            'text': tokenizer.decode(token_ids[start:end], skip_special_tokens=True),
            'turn_ids': owner_ids,
        })
        if end == len(token_ids):
            break
    return windows

windows = make_windows(flat_ids, owners)
for w in windows:
    print(f"WINDOW {w['window_id']} | turns {w['turn_ids']}\n{w['text']}\n")

## 3. Dense retrieval, then cross-encoder reranking

The same unchanged question is used for both stages, as in RAW-RAG. The paper retrieves 128 and retains 15; this tiny demo retrieves up to 6 and retains 3.

In [ ]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
question = example['question']

window_vectors = embedder.encode([w['text'] for w in windows], normalize_embeddings=True)
query_vector = embedder.encode([question], normalize_embeddings=True)[0]
dense_scores = window_vectors @ query_vector
candidate_count = min(6, len(windows))
candidate_ids = np.argsort(dense_scores)[::-1][:candidate_count]

pairs = [(question, windows[i]['text']) for i in candidate_ids]
cross_scores = reranker.predict(pairs)
ranked = sorted(
    [
        {'window': windows[int(i)], 'dense_score': float(dense_scores[i]), 'rerank_score': float(s)}
        for i, s in zip(candidate_ids, cross_scores)
    ],
    key=lambda x: x['rerank_score'],
    reverse=True,
)[:3]

for item in ranked:
    w = item['window']
    print(f"WINDOW {w['window_id']} | dense={item['dense_score']:.3f} | rerank={item['rerank_score']:.3f}")
    print(w['text'], '\n')

## 4. Expand selected windows to complete turns

Retrieval scores the windows, but the reader can receive the complete source turns overlapping those windows. This preserves dates, speakers, updates, and relations that a compressed memory record might lose.

In [ ]:
selected_turn_ids = sorted({
    turn_id
    for item in ranked
    for turn_id in item['window']['turn_ids']
})
evidence = '\n'.join(turns[i] for i in selected_turn_ids)
reader_prompt = f'''Answer the question using only the raw conversation evidence.
Resolve updates according to their timestamps.

Evidence:
{evidence}

Question: {question}
Answer:'''.strip()

print(reader_prompt)
print('\nExpected answer for this synthetic example:')
print(example['expected_answer'])

## Scope

This demonstrates the method's build → retrieve → rerank → evidence-assembly path. It does not reproduce the paper's Qwen3-4B or hosted-service configurations, benchmark-specific visibility rules, reader LLM, judge models, or reported accuracy. To approach the paper configuration, change the window/overlap to 256/32, retrieve 128, keep 15, and substitute the paper's embedding, reranking, and reader services.